# Trajectoires d’ajustement des finances publiques de la France, une

analyse avec debtwatch

Xavier Timbeau [](https://orcid.org/0000-0002-6198-5953) ([OFCE, Sciences Po Paris](http://www.ofce.fr))  
11 juillet 2025

In [ ]:
ofce::init_qmd()

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : 'scales'


L'objet suivant est masqué depuis 'package:purrr':

    discard


L'objet suivant est masqué depuis 'package:readr':

    col_factor


Le chargement a nécessité le package : sysfonts

Le chargement a nécessité le package : showtextdb

## 

En 2024, la France avait un déficit public de 5,8% du PIB et une dette publique de 113% du PIB. Les options politiques sont assez fermées, la réduction du déficit attendue en 2025 est assez faible (0,3 point de PIB) pour différentes raisons :

1.  un effort de consolidation budgétaire modéré (0,7 point de PIB),
2.  un effet multiplicateur qui réduit la croissance attendue à 0,4 point de PIB,
3.  la diminution de la contribution européenne (*NextGenEU*),
4.  un ralentissement de l’inflation, qui réduit la croissance nominale et la dépréciation de la dette publique,
5.  la hausse de la charge d’intérêt, liée au renouvellement de la dette à des taux (3,2% à 10 ans) nettement au dessus du taux moyen pondéré de la dette publique (1,94%).

Bien que la charge de la dette soit relativement faible, du fait de conditions d’endettement très favorables ces dernières années, les taux courants à long terme sont aujourd’hui plus élevés (3,32 % le 9 juillet 2025 pour l’OAT 10 ans) et un *spread* s’est creusé avec l’Allemagne (+0,63 % au 9 juillet). Ceci pourrait induire une hausse importante du taux apparent dans les prochaines années et combiné à un déficit qui reste élevé les finances publiques sont sous tension, parce que l’effort fiscal à consentir pour stabiliser la dette est important.

Le déficit structurel primaire, qui est une première évaluation de l’effort à réaliser pour stabiliser la dette publique, est en 2024 de 3,7 points de PIB (~110 milliards d’euros).

On se propose ici de préciser cet effort fiscal et d’apprécier les éléments qui peuvent l’alléger ou l’alourdir.

## Programme de Stabilité à Moyen Terme

Le point de départ de l’analyse est un compte macroéconomique à moyen terme. On utilise d’une part la prévision de printemps de l’OFCE pour les années 2025 et 2026, les éléments du PSMT de 2027 à 2029 (le rapport d’avancement du [PSMT d’avril 2025 - page 40](https://www.tresor.economie.gouv.fr/Articles/f6e70afa-7b97-4eaa-8ce3-892a4be60211/files/10caa42b-c3c8-44ca-8a6e-9847f4f55b34), qui diffère significativement de ce que le Conseil de l’Union Européenne a approuvé en [janvier 2025 - page 6/23](https://economy-finance.ec.europa.eu/document/download/f8be355b-dd72-4726-b81e-fea6e416fc1b_en?filename=MTFSP_2025_FR.pdf) détaillé dans le [PSMT initial d’octobre 2024 - page 68](https://www.tresor.economie.gouv.fr/Articles/0a792c43-0e8b-47f2-b8dc-048d8a1bf497/files/2b82dc6d-a2a8-479f-aæ2-0efc329cd113)) et qui pourrait être encore modifié depuis la suspension du PSC par la Commission le 4 juin 2025.

Au-delà de 2029, on prolonge les tendances observées et on détermine, au delà de 2029 la politique budgétaire pour stabiliser la dette à 110% du PIB. La politique budgétaire est donc restrictive (entre 0,9 et 0,6 point chaque année d’ajustement structurel primaire) jusqu’en 2029, pour un total de 3,8% du PIB d’ajustement, puis ce qu’il est nécessaire pour finaliser l’ajustement de façon à ce qu’en 2045 (2025+20), la dette publique soit de 110% et qu’elle soit stable à 110%. Comme on le verra plus loin, la cible de dette, l’ampleur de l’ajustement initial (de 2025 à 2029), le calendrier de l’ajustement ou encore les taux souverains modifient significativement la politique budgétaire nécessaire.

Le compte central est donné sur le @tbl-cc et il est basé sur les données de la Commission Européenne (AMECO) de printemps 2025. La croissance potentielle est fixée à 1,1% par an avec une baisse du NAIRU les 10 premières années, de 8 à 7% de la population active, qui place la croissance potentielle effective à 1,2% par an jusqu’en 2035,, puis 1,1% ensuite. L’inflation est supposée converger vers 1,75%. Le chômage d’équilibre est de 7%. La politique budgétaire suit le PSMT jusqu’en 2029, puis on applique une règle budgétaire linéaire d’ordre 1[1].

[1] La règle budgétaire a l’expression suivante : $ib = -0,05 \times ( d- d^*) + 0,5 \times (s-s^*) - 0,5 \times (og - 0)$

In [ ]:
cc <- simulations$cc |> 
  mutate(
    psmt_l = case_when(
      ibs == "psmt" ~ "impulsions optimales calculées à partir de 2030",
      ibs == "optimal" ~ "impulsions optimales calculées à partir de 2025",
      ibs == "psmtlr" ~ "impulsions selon une règle linéaire à partir de 2030",
      ibs == "lr" ~ "impulsions selon une règle linéaire à partir de 2025"),
      is_psmt = str_detect(ibs, "psmt"),
      is_lr = str_detect(ibs, "lr"), 
    nat = case_when(
      year<2025 ~"obsvervation",
      between(year, 2025, 2029)&is_psmt ~ "PSMT",
      between(year, 2025, 2029)&!is_psmt&is_lr ~ "règle linéaire" ,
      year>2029&is_lr ~ "règle linéaire",
      TRUE ~ "optimal"), 
    tooltip = glue("<b>{vlabels[variable]}</b><br><em>{psmt_l}</em><br>En {year} : {round(q0.5*100,1)}%"))

table_simulation(cc |> filter(ibs == "psmt"), debut = 2024, bck = 0)

,2024,2025,2026,2027,2028,2029,2030,LT (2050),Max. 25-50
Croissance du PIBa,"1,2","0,5","1,1","1,1","1,0","1,1","1,9","1,2",—
Ecart de productionb,"−0,7","−1,4","−1,5","−1,6","−1,7","−1,8","−1,0","0,0","c −1,8"
Croissance du PIB potentiela,—,"1,2","1,2","1,2","1,2","1,2","1,1","1,1",—
Inflationd,"2,3","1,3","1,4","1,5","1,5","1,5","1,4","1,7",—
Taux de chômagee,"7,4","8,1","8,9","9,0","9,0","9,0","8,5","7,0","9,0"
Solde publicb,"−5,8","−5,7","−5,1","−4,6","−3,9","−3,3","−3,1","−3,0","−2,2"
Solde public primaireb,"−3,6","−3,4","−2,6","−1,9","−1,2","−0,5","−0,2","−0,0","0,8"
Solde public structurel primaireb,"−3,3","−2,7","−1,9","−1,2","−0,4","0,3","0,3","−0,0","0,8"
Impulsion budgétaire primaireb,"0,1","−0,8","−0,9","−0,7","−0,8","−0,7","0,0","0,0",—
dont POb,"0,4","−0,4","−0,4","−0,3","−0,4","−0,4","0,0","0,0",—


In [ ]:
annotations <- tribble(
  ~variable, ~x, ~y, ~texte, ~ibs, ~hjust,
  "dettep", 2037, 1.16, "optimal à partir de 2030", "psmt", 0,
  "dettep", 2035, 1.11, "règle linéaire à partir de 2030", "psmtlr", 1)
data_cc <- cc |> 
  filter(
    ibs %in% c("psmtlr", "psmt"),
    between(year, 2024, 2050), 
    variable %in% c("og", "ib", "dettep", "sep")) |> 
  mutate(ibs = factor(ibs, c("psmtlr", "psmt")))

gg <- ggplot(data_cc) +
  PrettyCols::scale_color_pretty_d("Joyful") +
  PrettyCols::scale_fill_pretty_d("Joyful") +
  geom_ribbon(
    aes(x=year, ymin=q0.025, ymax=q0.975, fill = ibs), alpha = 0.05) +
  geom_line(
    aes(x = year, y = mean, color = ibs), linewidth = 0.25) +
  geom_text(
    data=annotations, 
    aes(x = x, y = y, label = texte, color = ibs, hjust = hjust), size = 2 ) +
  geom_point_interactive(
    aes(x = year, y = mean, fill = ibs, shape = nat, size = nat,
        tooltip = tooltip, data_id = year), 
     color="white", size = 1., shape = 21, stroke=0.25, hover_nearest = TRUE)+
  scale_size_manual(values= c("obs" = 2, "pmst" = 1.5, "rule" = 1)) +
  scale_shape_manual(values= c("obs" = 21, "pmst" = 23, "rule" = 23)) +
  facet_wrap(ggplot2::vars(variable), scales = "free_y", labeller = labeller(variable= vlabels)) +
  scale_y_continuous(labels = scales::label_percent(.1))+
  scale_x_continuous(
    breaks = scales::pretty_breaks(), 
    labels = date1,
    minor_breaks = scales::breaks_width(1)) +
  xlab(NULL)+ylab(NULL)+
  theme_ofce()+
  guides(shape = "none", size = "none", fill="none", color = "none",
         x = guide_axis(minor.ticks = TRUE))+
  labs(color = NULL, fill=NULL)

girafy(gg, r=1.5)

La hausse de la dette publique est modérée mais nette, bien que restant juste en dessous de 120%, sous l’effet de la réduction du déficit planifiée par le PSMT. Du fait des politiques restrictives, l’écart de production augmente au fur et à mesure des impulsions négatives. L’impulsion est partagée en deux entre les PO et les dépenses, le multiplicateur de PO est de 0,5, celui de dépense de 0,8, ce qui induit un multiplicateur moyen de de 0,65.

Une fois le gros de l’ajustement réalisé, en 2030, une légère impulsion positive est possible ce qui maintient le solde public primaire en deçà de -0,9% du PIB. Comme le taux apparent (ou le taux moyen pondéré de la dette publique) est inférieur à la croissance nominale, la dette baisse malgré un déficit primaire, cette baisse se réduisant au fur et à mesure de la hausse de la charge d’intérêts et évoluant en fonction du solde structurel.

La charge d’intérêts augmente substantiellement (de 0,6 point entre 2025 et 2030) sous l’effet de la hausse du taux moyen pondéré, poussé par le taux courant qui tend vers 3,0% à terme[1]. La valeur de 3,2% pour le taux courant est la somme de la croissance potentielle en volume et de l’inflation. Cela correspond donc à $r-g=0$ à long terme. Cette hausse est moins rapide que celle projetée par le FMI dans le [WEO d’avril 2025](https://www.imf.org/en/Publications/WEO/Issues/2025/04/22/world-economic-outlook-april-2025#Chapters) qui retient pour la France une hausse de 1,5 points de PIB entre 2025 et 2030 – ce qui paraît largement exagéré, mais non sans conséquences sur l’analyse des finances publiques de la France à moyen terme.

A plus long terme, une faible impulsion négative suffit à ramener lentement le solde primaire vers 0 tandis que la charge d’intérêt tend vers 3.6 %. Cette charge d’intérêt cible est longue à atteindre compte tenu de la formulation de l’équation de renouvellement de la dette qui est auto régressive et non pas à générations (elle n’inclue pas non plus d’éléments de variation de la maturité de la dette en fonction de la pente de la courbe des taux ni l’effet de l’indexation d’une part des OAT sur l’inflation).

L’écart de production retenu est de -0,7 point de PIB, en suivant le scénario macroéconomique du rapport d’avancement annuel. En conséquence, le déficit public structurel primaire est -3,7% du PIB en 2024 et de 3% du PIB en 2025. L’effort total à réaliser (en incluant 2025) pour stabiliser la dette publique (sous l’hypothèse d’un écart critique nul) et donc de 3,7 points de PIB naïvement, soit proche de ce qui est inscrit dans le PSMT.

La Commission Européenne fournit une évaluation différente de l’écart de producion, l’affichant à une valeur proche de 0 en 2024 dans la version du printemps 2025 de la base AMECO. Selon AMECO, le déficit structurel primaire est donc plus élevée de 0,4 point de PIB.

## Politique budgétaire optimale dans *debtwatch*

Une politique budgétaire optimale est calculée dans *debtwatch* 2.0, en utilisant une approche différente de la version 1.

Le principe est simple : les impulsions budgétaires successives (disons de 2025 à 2055) sont déterminées de façon à ce que la dette publique soit égale à la cible de dette à partir d’une date donnée (dans le cas central 2045, 20 ans après 2025). Ce problème admet une infinité de solutions, puisque il suffit que le déficit primaire soit nul à partir de 2045 et que la dette en 2045 soit égale à la cible. On choisit parmi toutes ces solutions celle qui minimise l’écart quadratique d’activité ou des impulsions. Formellement la fonction de perte est quadratique et s’écrit :

$$
\mathcal{L} = \lambda_{og}\sum_{t=2025}^{\infty} \beta^t og_t^2 + \lambda_{ib} \sum_{t=2025}^{\infty} \beta^t ib_t^2
$$ {#eq-perte-quad}

$\lambda_{og}$ et $\lambda_{ib}$ permettent de donner un poids relatif aux écarts de production au carré et aux impulsions. $\beta$ est un facteur d’actualisation. La fonction de perte est dérivable et permet l’utilisation d’algorithmes numériques standards de minimisation. L’interprétation est soiot la minimisation de la « variance » des écarts de production, c’est-à-dire une trajectiore qui s’écarte le moins possible de de 0 et qui donne un poids aux trajectoires où l’écart est le plus constant possible. On minimise l’impact de la politique restrictive. La partie en $ib_t^2$ est l’équivalent pour les instruments et permet de représenter un monde où il est difficile de modifier le budget. Cette inertie si elle n’est pas rationnelle est réaliste. Le mélange entre les deux fonctions de perte conduit à associer les deux arguments. Le facteur d’actualisation consiste à préférer des conséquences lointaines plutôt qu’immédiates et constitue une façon simple de justifier la *backloading* par rapport au *frontloading*.

Alternativement, on utilise la fonction de perte @eq-perte-min, où c’est le creux d’écart de production que l’on minimise. Cette fonction de perte conduit, comme on le verra plus loin, à produire des trajectoires dont la partie $og_t$ négative est la plus limitée possible. Cette fonction de perte induit un assez grand backloading, par construction. Pour faciliter le calcul numérique, on utilise une approximation continue de la fonction $max$[2].

$$
\mathcal{L} = max(-og_t)
$$ {#eq-perte-min}

Le programme que l’on résout est alors, $\mathcal{S}$ étant la solution du système dynamique qui permet de calculer la trajectoire de la dette à partir des impulsions (et dont les équations sont en annexe) :

$$
\min_{ib_{t}}\mathcal{L} \\
\text{sous contrainte : pour } t\geq T \text{  } d_t = d^* \\
\text{avec } d_t = \mathcal{S}(ib_t)
$$ {#eq-prog-min}

Lorsqu’on intègre le PSMT, la politique budgétaire optimale est calculée à partir de 2030 et les impulsions du PSMT sont utilisées de 2025 à 2029. La cible de dette joue à partir de $T$ (en pratique $T=2045$).

Parce que les impulsions modifient le déficit structurel, une impulsion opérée plus tôt dans le temps aura un effet sur la dette (qui est l’accumulation des déficits) plus élevé, s’accumulant chaque année. Cette arithmétique pousse à faire le plus tôt possible les impulsions négatives afin de réduire le déficit structurel. Néanmoins, la minimisation de la fonction de perte conduit à repartir l’effort sur plusieurs périodes afin de limiter les effets induits.

La chaîne causale qui part des contrôles (les impulsions) vers la variable d’état « dette » est simple mais incorpore quelques mécanismes importants : par l’effet multiplicateur, l’impulsion perturbe l’écart de croissance, qui revient ensuite à l’équilibre ($og=0$) progressivement. L’écart de croissance défini, avec l’impulsion, le solde effectif, qui induit la dynamique de la dette. L’écart de croissance joue sur le chômage (équation d’*Okun*), qui perturbe l’inflation (*Phillips*) de sa cible de long terme. Le taux d’intérêt apparent dépend du taux passé (renouvellement de la dette partiel) et du taux courant. Le taux courant est défini comme une prime de risque sur le taux « neutre » égal à la croissance potentielle. Lorsque l’écart de croissance se creuse, l’inflation ralentie et le taux courant s’abaisse. La dynamique de la dette intègre donc un effet solde, un effet croissance en volume, un effet inflation et un effet taux (courant/apparent), ce qui induit une non linéarité en particulier lorsque il existe un écart entre le taux apparent et le taux de croissance nominale. Enfin, possiblement, une boucle de rétroaction fait varier le taux en fonction des évolutions de la dette (en écart à une dette de référence) et le multiplicateur peut dépendre du cycle (i.e. de l’écart de croissance). Les équations du modèle se trouvent en annexe.

Les variables d’état du modèle (qui permettent de décrire complètement à un instant donné l’état du modèle) sont : la dette publique (en point de PIB), le solde public, l’écart critique[3], l’écart de production, le taux de croissance du prix du PIB (en écart à la cible) et le taux d’intérêt instantané à long terme. Les variables de contrôle sont les impulsions budgétaires à chaque période.

Bien que simple, cette dynamique contient suffisamment d’ingrédients pour produire des résultats intéressants, en particulier pour des conditions initiales éloignées de l’équilibre. Dans la version standard du modèle, les coefficient des équations dynamiques sont des constantes, et donc le modèle dynamique est quasi-linéaire si l’écart critique reste faible. Cette hypothèse peut être facilement levée en introduisant des équations qui font dépendre ces différents coefficients des variables d’état du modèle.

La version 1 de *debtwatch* utilisait un algorithme moins gourmant en calcul : en fonction des paramètres principaux du modèle, un règle budgétaire linéaire[4] était estimée (de façon à minimiser une fonction de perte identique) et appliquée aux différentes conditions initiales du système. Une telle règle d’ordre 1 garantit l’équilibre à long terme (la dette publique converge vers sa cible, à la vitesse choisie), mais n’est qu’une approximation de la politique optimale. Dans *debtwatch* 2, un algorithme de minimisation numérique est utilisé pour déterminer les impulsions budgétaires qui minimisent la fonction de perte sous contrainte. L’algorithme utilisé est COBYLA[5] \[@powell1994\] dans son implémentation dans le package R `{nloptr}` qui utilise la bibliothèque Nlopt \[@NLopt, @nloptr\]. L’algorithme du Lagrangien Augmenté \[@conn1991\] peut être employé et donne des résultats très proches. Une alternative, plus coûteuse en temps de calcul, est l’algorithme ISRES[6] qui permet de mieux identifier les minima globaux. Ces algorithmes évaluent le gradient et les matrices hessiennes du problème (fonction de perte et contraintes) numériquement. Cette approche permet de calculer numériquement la politique optimale ainsi définie et fonctionne également avec une version du modèle (fortement) non linéaire.

On ajoute à cette politique budgétaire optimale une règle budgétaire (comme dans *debwatch* 1), mais qui n’intervient que pour les déviations par rapport au compte utilisé pour le calcul de la règle budgétaire optimale. Cela permet de traiter les perturbations stochastiques ajoutées au modèle et d’adapter la politique budgétaire sans refaire une optimisation complète.

Le @fig-optim affiche les impulsions budgétaires pour les différentes fonction de pertes envisagées pour une politique optimale menée dès 2025. Les impulsions cumulées sont toutes égales au bout du compte, parce que la cible de dette et le point de départ sont égaux. Le modèle dynamique est linéaire et autorégressif, ce qui conduit à cette propriété. La valeur du cumul (en commençant en 2025) est de -3,6%, ce qui constitue l’ajustement structurel nécessité pour stabiliser la dette à 110%. La différence avec le solde structurel primaire en 2025 est l’accumulation des intérêts au dessus de 110% dans la phase de réduction de la dette.

Comme les impulsions sont cumulées égales à la fin de la période – le modèle dynamique considéré est linéaire –, il en est de même pour les écarts de production cumulés. Ainsi, la fonction de perte conditionne le calendrier des impulsions.

La fonction de perte *og* (somme des carrés des écarts de production) produit une impulsion fortement positive en 2030 et conduit à l’accumulation d’écarts de production un peu moins marquée entre 2030 et 2034. Ensuite, pour cette fonction de perte, des impulsions négatives sont nécessaires et creusent l’écart de production. Les écarts de production négatifs sont plus répartis que pour d’autres fonctions de perte.

La fonction de perte *ib* (somme des impulsions au carré) est à l’opposé de *og (*somme des écarts de production au carré). Au lieu de minimiser la résultante, elle conduit à minimiser l’instrument. Le cumul des impulsions est donc moindre par rapport au point de départ induit par le PSMT. L’écart de production continue donc de se creuser dans la suite du PSMT, puis se résorbe ensuite conduisant à un profil moins étalé pour le cumul des écarts de production.

La minimisation de la somme des écarts négatifs de production (*sog*) induit un profil presque carré pour l’écart de production. Le critère « minimum d’écart » partage avec la fonction de perte *og* (somme des écarts au carré) de produire une impulsion positive en 2030. Par rapport à l’objectif de stabiliser la dette à 110%, le PSMT surajuste ce qui est corrigé dans ce cas par cette forte impulsion positive. Cela permet d’atteindre l’écart minimal (de l’ordre de -0.2%). Comme l’impulsion initiale est plus positive que pour les autres fonctions de perte, il est ensuite nécessaire de compenser par des impulsions plus négatives ensuite.

[1] On suppose que le taux de marché à 10 ans tend vers le taux de croissance nominal ($r=g$)pour le compte de référence. D’autres hypothèses sont envisagées dans la suite.

[2] par une fonction $max(x_1,x_2) \approx (x_1^\sigma + x_2^\sigma)^{1/\sigma}$ avec $sigma$ assez grand (en pratique on prendra 8.

[3] L’écart critique est égal à $(r-g)/(1+g)$ et résume les variables taux d’intérêt nominal $r$, croissance du PIB en valeur $g$. Compte tenu de la frome des équations

[4] La règle est linéaire en fonction des variables d’état du système (écart de production, dette, solde, écart critique, écart d’inflation à la cible) exprimées en fonction de leur écart aux valeurs d’équilibre.

[5] Constrained Optimization BY Linear Approximations.

[6] Improved Stochastic Ranking Evolution Strategy

In [ ]:
rlabels <- c(lr = "Règle linéaire", og = "Ecarts de production au carré",  ib = "Impulsions au carré", mo = "minimax de l'opposé des écarts production")
vlabs <- c(ibc = "Impulsion", ogc = "Ecart de production", ib = "Impulsion", og = "Ecart de production")
doptim <- simulations$pertes |> 
  filter(between(year, 2025, 2050),
         perte %in% c("lr", "ib", "og", "mo")) |>
  pivot_longer(c(og, ogc, ib, ibc), names_to = "variable") |> 
  mutate(
    cumul = ifelse(str_detect(variable, "c$"), "c", "nc"),
    variable = str_sub(variable, 1,2)  ) |> 
  pivot_wider(names_from = cumul, values_from = value) |> 
  group_by(variable, df, perte) |> 
  arrange(year) |> 
  ungroup() |> 
  pivot_longer(cols = c(c, nc)) |> 
  mutate(
    cum = case_when(
      name == "c" ~ "Cumulé",
      name == "nc" ~ "Non cumulé" )  ) |> 
  pivot_wider(names_from = quantile, values_from = value) |> 
  mutate(
    perte = factor(perte, c("lr", "og", "ib", "mixte", "mo")),
    dfl = ifelse(perte!="lr", str_c(" ; \u03b2=", round(100-100*df),"%"),""),
    tooltip = glue("<b>{rlabels[perte]}{dfl}</b><br>{year}<br>{vlabs[variable]} : {round(mean*100,1)}%"))
dpsmt <- cc |> filter(variable=="ib",ibs == "psmt", year>=2025, year<=2029)

ggo <- function(x) {
  ggplot(doptim |> filter(year>=2025, variable == "ib", cum == "Non cumulé", dstar == x )) +
    PrettyCols::scale_color_pretty_c(
    "Blues",
    breaks = c(0, 0.05, 0.1), labels=c("100%", "95%", "90%"), name = "\u03b2") +
  PrettyCols::scale_fill_pretty_c("Blues") +
  # geom_line(
  #   data = cc |> filter(variable=="ib",ibs == "psmt", year>=2025, year<=2029),
  #   aes(x = year, y = mean), color = "grey80", size=.25) +
  geom_hline(yintercept = 0, linewidth = 0.1) +
  geom_ribbon(aes(x=year, ymin=q0.025, ymax=q0.975, group=df, fill = df), alpha=0.1) +
  geom_segment(data = dpsmt, aes(x=year, yend=mean, y=0, xend=year), color="orange", linewidth=0.1)+
  geom_point_interactive(
    data = dpsmt,
    aes(
      x = year, y = mean, data_id = year,
      tooltip = glue(
        "<b>Programme de stabilité à moyen terme (PSMT)</b><br>{year}<br>Impulsion: {round(mean*100)}%")), 
    hover_nearest = TRUE,  size = .25, color = "orange") +
    geom_line(
      aes(x = year, y = mean, color = (df), group=df))+
    geom_point_interactive(
      aes(x = year, y = mean, fill = df, group=df,
          tooltip = tooltip, data_id = year), 
      color="white", shape=21, size = 1, stroke=0.25, hover_nearest = TRUE)+
    scale_size_manual(values= c("obs" = 0, "pmst"=0, "rule" = 1.5)) +
    scale_shape_manual(values= c("obs" = 21, "pmst"=21, "rule" = 21)) +
    facet_wrap(
      vars(perte),
      labeller = as_labeller(rlabels)) +
    scale_y_continuous(
      labels = scales::label_percent(.1), 
      limits = c(-0.015, 0.025), 
      oob = scales::squish)+
    scale_x_continuous(
      breaks = scales::pretty_breaks(),
      labels = date1,
      minor_breaks = scales::breaks_width(1)) +
    guides(
      x = guide_axis(minor.ticks=TRUE) )+
    xlab(NULL)+ylab(NULL)+
    theme_ofce()+
    guides(shape = "none", fill="none", size = "none", color = guide_legend(nrow=1))+
    labs(color = NULL, fill=NULL) +
    theme(legend.position.inside = c(0.0, 0.36),
          legend.justification = c(0,1),
          legend.direction = "horizontal") +
    ofce_caption(
      source = "debtwatch 2.0, AMECO printemps 2025",
      note = "Les impulsions sont calculées suivant une politique budgétaire optimale (i.e. minimisant la fonction de perte considérée) et aboutissant à {round(x*100)}% de dette en 2045 à partir de 2025.")
}
dstars <- distinct(doptim, dstar) |> pull(dstar)
names(dstars) <- str_c("d*=", round(dstars*100),"%")
tabset <-  map(dstars, ~ ggo(.x))

tabsetize(tabset)

On peut comparer la règle linéaire aux règles optimales. La première différence est que la règle linéaire ne respecte pas strictement la contrainte de dette (il faut attendre au delà de la date limite pour que la dette atteingne la cible). Ceci conduit à un moindre cumul d’impulsions négatives que pour les règles optimales, la correction étant reportée à plus tard.

L’utilisation d’un facteur d’actualisation donne plus de poids au présent par rapport aux périodes plus lointaines. Cela ne change pas radicalement les trajectoires si ce n’est de retarder le creusement de l’écart de production et de procéder à du *backloading.* Ainsi, pour un facteur d’actualisation de 90%, l’impulsion initiale est suivant les règles nettement diminuée. Lorsque la fonction de perte est la somme des écarts de production au carré, l’impulsion en 2025 est positive, reportant l’ajustement à plus tard.

Dans la suite nous utiliserons la fonction de perte *ib* (somme des écarts de l’impulsion au carré), avec un facteur d’actualisation égal à 100%.

## Un ajustement optimal moindre que l’évaluation naïve

In [ ]:
table_simulation(cc |> filter(ibs == "optimal"), debut = 2024, bck = 0)

,2024,2025,2026,2027,2028,2029,2030,LT (2050),Max. 25-50
Croissance du PIBa,"1,2","0,7","1,2","1,2","1,2","1,2","1,2","1,3",—
Ecart de productionb,"−0,7","−1,3","−1,2","−1,2","−1,2","−1,2","−1,1","0,0","c −1,3"
Croissance du PIB potentiela,—,"1,2","1,2","1,2","1,2","1,2","1,1","1,1",—
Inflationd,"2,3","1,3","1,4","1,5","1,5","1,5","1,5","1,7",—
Taux de chômagee,"7,4","8,0","8,7","8,7","8,6","8,5","8,4","7,1","8,7"
Solde publicb,"−5,8","−5,7","−5,2","−4,9","−4,5","−4,1","−3,7","−3,0","−1,8"
Solde public primaireb,"−3,6","−3,5","−2,8","−2,2","−1,7","−1,2","−0,8","−0,0","1,2"
Solde public structurel primaireb,"−3,3","−2,9","−2,2","−1,7","−1,2","−0,7","−0,2","−0,0","1,3"
Impulsion budgétaire primaireb,"0,1","−0,6","−0,7","−0,5","−0,5","−0,5","−0,5","0,1",—
dont POb,"0,4","−0,3","−0,3","−0,3","−0,2","−0,2","−0,2","0,0",—


Le @tbl-ccoptim découle de l’application de la politique budgétaire optimale à la place de ce que le PSMT anticipe. L’objectif est de stabiliser la dette publique à 110% du PIB (ce qui est peut être un objectif moins ambitieux que celui du PSMT), en minimisant le recours aux impulsions (moindre variation de la politique budgétaire). A l’horizon de l’année 2029, au lieu d’un ajustement de 3,8% point de PIBn, la politique budgétaire optimale – sous les conditions énoncées, cible de dette et fonction de perte – est un ajustement de 2,8 points de PIB (80 milliards d’euros). L’effet sur l’activité est minoré par rapport au PSMT et le maximum de la dette publique est supérieur (121,5 % du PIB contre 120% du PIB dans le PSMT \[@tbl-cc\]).

A long terme, la totalité de l’ajustement est de 3,4 point de PIB. C’est un peu moins que le solde structurel de 2024 (3,7 point de PIB) principalement parce que ce calcul intègre l’effet du taux apparent inférieur à la croissance nominale à) partir de 2026 (lignes taux souverain pondéré, croissance du PIB, inflation du @tbl-ccoptim). Comme on le verra plus loin, cet ajustement de long terme est presque indépendant de la cible de dette, du moins tant que l’écart entre le taux pondéré et le taux de croissance nominal tend vers 0). Cette propriété découle de la quasi-linéarité du modèle (et s’estompe lorsque l’écart critique est non nul) et le choix d’une cible de dette joue sur le calendrier des impulsions – plus la cible de dette est restrictive, plus les impulsions doivent être négatives tôt, mais sans que cela ne change l’ajustement total.

In [ ]:
annotations <- tribble(
  ~variable, ~x, ~y, ~texte, ~ibs, ~hjust,
  "dettep", 2037, 1.16, "optimal à partir de 2025", "optimal", 0,
  "dettep", 2035, 1.11, "PSMT", "psmt", 1)

data_cc <- cc |> 
  filter(
    ibs %in% c("psmt", "optimal"),
    between(year, 2024, 2050), 
    variable %in% c("og", "ib", "dettep", "sep")) |> 
  mutate(ibs = factor(ibs, c("psmt", "optimal")))

gg <- ggplot(data_cc) +
  PrettyCols::scale_color_pretty_d("Joyful") +
  PrettyCols::scale_fill_pretty_d("Joyful") +
  geom_ribbon(
    aes(x=year, ymin=q0.025, ymax=q0.975, fill = ibs), alpha = 0.05) +
  geom_line(
    aes(x = year, y = mean, color = ibs), linewidth = 0.25) +
  geom_text(
    data=annotations, 
    aes(x = x, y = y, label = texte, color = ibs, hjust = hjust), size = 2 ) +
  geom_point_interactive(
    aes(x = year, y = mean, fill = ibs, shape = nat, size = nat,
        tooltip = tooltip, data_id = year), 
     color="white", size = 1., shape = 21, stroke=0.25, hover_nearest = TRUE)+
  scale_size_manual(values= c("obs" = 2, "pmst" = 1.5, "rule" = 1)) +
  scale_shape_manual(values= c("obs" = 21, "pmst" = 23, "rule" = 23)) +
  facet_wrap(ggplot2::vars(variable), scales = "free_y", labeller = labeller(variable= vlabels)) +
  scale_y_continuous(labels = scales::label_percent(.1))+
  scale_x_continuous(
    breaks = scales::pretty_breaks(), 
    labels = date1,
    minor_breaks = scales::breaks_width(1)) +
  xlab(NULL)+ylab(NULL)+
  theme_ofce()+
  guides(shape = "none", size = "none", fill="none", color = "none",
         x = guide_axis(minor.ticks = TRUE))+
  labs(color = NULL, fill=NULL)

girafy(gg, r=1.5)

## Sensibilité de l’ajustement

Le @fig-sensib présente l’analyse de sensibilité à trois paramètres essentiels du modèle. Le premier est la cible de dette, $d^*$, le second la valeur critique $ec^*$ et enfin le troisième concerne le calendrier : on compare d’un côté l’application du PSMT puis d’une politique permettant la stabilisation de la dette à sa cible, de l’autre l’application dès 2025 de la politique optimale.

Le @fig-sensib présente pour chaque combinaison (donc 3 $\times$ 2 $\times$ 2 possibilités) les trajectoires cumulées des impulsions budgétaires depuis 2029 jusqu’à 2045. Pour chaque graphique est indiqué le cumul total (jusqu’en 2045) des impulsions et la valeur cumulée maximale de l’ajustement (soit le minimum des impulsions négatives, l’impulsion étant l’opposé de l’ajustement).

Plus la cible est stricte, plus l’ajustement est important. Plus la prime de risque sur la dette française est élevée, plus l’ajustement nécessaire est important. Entre une prime négative de -0,5% et nulle, l’écart d’ajustement est d’un peu plus de 0,5 point de PIB (16 milliards d’euros).

In [ ]:
library(ggh4x)

bo <- sourcoise::sourcoise("ajust/simulations ajustement bigone.R")

gg <- bo$trajectoires |> 
  filter(year>=2029, year<=2050, mult=="fixe", ibs %in% c("optimal", "psmt"),
         dstar %in% c("d*=90%", "d*=110%", "d*=120%"), infstar == 0.0175,
         ecstar %in% c(-0.005, 0), variable == "ibc") |> 
  mutate(tooltip = glue("<b>{year}</b><br>Cumul d'impulsion: {round(mean*100,2)}%")) |> 
  ggplot() +
  aes(y = mean, ymin=q0.25, ymax =q0.75, 
      x = year) +
  geom_ribbon(aes(ymin=q0.25, ymax=q0.75),
              alpha = 0.2, fill = "orange") +
  geom_line(
    alpha = 1, size = 0.1, linewidth = 0.2, color = "orange") +
  geom_point_interactive(
    aes(tooltip=tooltip, data_id = year),
    data = ~filter(.x, year%in%c(2029, 2045, 2050)), size = 0.75, color = "orange") +
  geom_text(
    aes(label=libc, y=-0.064),
    color = "black",
    size = 2, hjust = 1 )+
  # scale_color_distiller(palette="Oranges", direction = -1, name = NULL)+
  scale_y_continuous(
    limits = c(-0.07, -0.02),
    breaks=scales::breaks_width(0.01),
    labels=scales::label_percent(1),
    oob = scales::squish)+
  scale_x_continuous(breaks = c(2030, 2040, 2050), 
                     labels = c("2030", "40", "50"),
                     minor_breaks = scales::breaks_width(1)) +
  guides(color = guide_legend(nrow = 1),
         x = guide_axis(minor.ticks = TRUE)) +
  theme_ofce(
    legend.position = "bottom",
    legend.direction = "horizontal",
    legend.key.width = unit(24, "pt"),
    legend.text.position = "bottom",
    axis.text.x = element_text(hjust=0.5, size = rel(0.9)),
    legend.title = marquee::element_marquee(vjust = 0.9),
    panel.spacing = unit(5, "pt")
  )+
  facet_nested(
    cols=vars(ibs, dstar), rows= vars(ecstar),
    labeller = labeller(
      ecstar = c("-0.005" = "ec*=-0.5%", 
                 "0" = "ec*=0%", 
                 "0.005" = "ec*=0.5%"),
      infstar = c("0.0175"="i*=1,75%", "0.02"="i*=2.0%"),
      ibs = c("psmt"= "PSMT jusqu'en 2029 puis optimal", "optimal"="Optimal depuis 2025", "Frontload", "Backload"))) +
  xlab(NULL)+
  ylab("Cumul des impulsions")

ℹ Please use `linewidth` instead.

(`geom_text()`).

Pour la colonne de droite (*PSMT jusqu’en 2029 puis optimal*), le point de départ est le même dans toutes les configurations, puisque c’est le point de sortie du PSMT en 2029. La comparaison avec la colonne de gauche donne un premier enseignement : le PSMT sur ajuste sauf dans le cas d’une cible de dette à 90% et une prime de risque de la France de +0,5%. Dans ce cas là, l’ajustement à réaliser est le plus important. Le PSMT conduit à surajuster de presque 2 point dans le cas le plus favorable, à savoir une prime de risque de -0,5% et une cible de dette de 120%. Le surajustement en 2029 est de plus d’un point de PIB (30 milliards d’euros) pour une prime nulle et une cible de 110%.

Le @fig-ajustement résume le lien entre l’ajustement nécessaire en 2029 et au total en fonction de la cible de dette ainsi que la sensibilité de cette valeur à l’écart critique. Le PSMT correspond à la politique optimale si la dette publique visée en 2045 est 97,5% et que la prime de risque est de 0,5% ou une cible de dette de 90% et une prime de risque de -0,5%. L’ajustement final dépend de la cible de dette du fait des non linéarités et de l’effet boule de neige sur la dette (négatif initialement, puis dépendant de l’écart critique).

In [ ]:
lprd <- c(psmt ="cumul de 2025 à 2029", full = "cumul de 2025 à 2050")
libs <- c(optimal = "Optimal depuis 2025", psmt = "PSMT jusqu'en 2029, puis optimal")
ga <- bo$mins |>
  filter(variable == "cumib", period %in% c("full", "psmt"), 
         ibs %in% c("psmt", "optimal"), ecstar %in% c(-0.005, 0, 0.005), infstar == 0.0175)  |> 
  mutate(period = factor(period, c("psmt", "full")),
         tooltip = glue("<b>{lprd[period]}</b><br>{libs[ibs]}<br>d*={round(dstar*100,1)}% \u03a3(ib)={round(mean*100,2)}%")) |> 
  ggplot() +
  aes(x=dstar, y = mean, color=ibs, group=ibs)+
  geom_ribbon(aes(ymin = q0.025, ymax = q0.975), alpha =0.1, color = "transparent") + 
  scale_colour_pretty_d("Joyful", name = NULL, labels = libs) +
  scale_fill_pretty_d("Joyful", name = NULL, labels = libs) +
  geom_line()+
  scale_y_continuous(labels = label_percent(1)) +
  scale_x_continuous(labels = label_percent(1)) +
  ylab("Cumul de l'impulsion")+
  xlab("d*")+
  geom_point_interactive(
    aes(tooltip=tooltip, data_id = dstar, fill = ibs),
    size = 1, shape = 21, color = "grey90") +
  guides(color = guide_legend(title = NULL), fill = guide_legend(title = NULL))+
  theme_ofce(
    legend.position.inside = c(0.02, 0.98)
  )+
  facet_grid(rows = vars(ecstar), cols = vars(period),
             labeller = labeller(period = lprd, ecstar = c("0" = "ec*=0%", "-0.005"="ec*=-0,5%", "0.005"="ec*=0,5%")))
   
girafy(ga)

Le @fig-sensibminog représente le lien entre l’ajustement nécessaire et le creusement de l’écart de production induit. Cet exercice repose sur une valeur moyenne du multiplicateur de 0,65 la première année, et serait plus marqué pour un multiplicateur plus important. Le surajustement du PSMT se traduit par un creux plus profond de 0,3 point de PIB ou de façon équivalente 0,3 point de chômage.

In [ ]:
lprd <- c(psmt ="cumul de 2025 à 2029", full = "entre 2025 à 2050")
libs <- c(optimal = "Optimal depuis 2025", psmt = "PSMT jusqu'en 2029, puis optimal")
gminog <- bo$mins |>
  filter(variable == "minog", period %in% c("full"), 
         ibs %in% c("psmt", "optimal"), ecstar %in% c(0), infstar == 0.0175)  |> 
  mutate(
    period = factor(period, c("psmt", "full")),
    tooltip = glue(
      "<b>{lprd[period]}</b><br>{libs[ibs]}<br>d*={round(dstar*100,1)}%
      min(og)={round(mean*100,2)}%")) |> 
  ggplot() +
  aes(x=dstar, y = mean, color=ibs, group=ibs)+
  geom_ribbon(aes(ymin = q0.025, ymax = q0.975), alpha =0.1, color = "transparent") + 
  scale_colour_pretty_d("Joyful", name = NULL, labels = libs) +
  scale_fill_pretty_d("Joyful", name = NULL, labels = libs) +
  geom_line()+
  scale_y_continuous(labels = label_percent(.1)) +
  scale_x_continuous(labels = label_percent(1)) +
  ylab("Minimum de l'écart de production")+
  xlab("d*")+
  geom_point_interactive(
    aes(tooltip=tooltip, data_id = dstar, fill = ibs),
    size = 1, shape = 21, color = "grey90", hover_nearest = TRUE) +
  guides(color = guide_legend(title = NULL), fill = guide_legend(title = NULL))+
  theme_ofce(
    legend.position.inside = c(0.02, 0.98)
  )
   

gmaxd <- bo$mins |>
  filter(variable == "maxd", period %in% c("full"), 
         ibs %in% c("psmt", "optimal"), ecstar %in% c(0), infstar == 0.0175)  |> 
  mutate(period = factor(period, c("psmt", "full")),
         tooltip = glue("<b>{lprd[period]}</b><br>{libs[ibs]}<br>d*={round(dstar*100,1)}% max(dette)={round(mean*100,2)}%")) |> 
  ggplot() +
  aes(x=dstar, y = mean, color=ibs, group=ibs)+
  geom_ribbon(aes(ymin = q0.025, ymax = q0.975), alpha =0.1, color = "transparent") + 
  scale_colour_pretty_d("Joyful", name = NULL, labels = libs) +
  scale_fill_pretty_d("Joyful", name = NULL, labels = libs) +
  geom_line()+
  scale_y_continuous(labels = label_percent(1)) +
  scale_x_continuous(labels = label_percent(1)) +
  ylab("Maximum de dette")+
  xlab("d*")+
  geom_point_interactive(
    aes(tooltip=tooltip, data_id = dstar, fill = ibs),
    size = 1, shape = 21, color = "grey90", hover_nearest = TRUE) +
  guides(color = guide_legend(title = NULL), fill = guide_legend(title = NULL))+
  theme_ofce(
    legend.position.inside = c(0.02, 0.98)
  )
library(patchwork)
gg <- gminog / gmaxd

girafy(gg)

## Si l’écart de production est très persistant

In [ ]:
ogmce <- sourcoise::sourcoise("ajust/simulations ajustement mceog.r")

ajustl <- c("0.1"="ajustement lent", "0.3" = "ajustement moyen", "0.5"="ajustement rapide")

gminog <- ogmce$trajectoires |> 
  filter(year>=2025, year<=2050, mult=="fixe", ibs %in% c("optimal"),
         dstar %in% c("d*=110%"), og_mce %in% c(0.1, 0.3, 0.5),
         ecstar %in% c(0), variable %in% c("ibc", "og")) |> 
  mutate(
    ajustl = ajustl[str_c(og_mce)],
    tooltip = glue(
  "<b>{year}</b>
  multiplicateur à 1 an: {og_dep} ; {ajustl} ({og_mce})
  Cumul d'impulsion: {round(mean*100,2)}%")) |> 
  ggplot() +
  aes(y = mean, ymin=q0.25, ymax =q0.75, x = year, color =  og_dep, group=og_dep) +
  # geom_ribbon(aes(ymin=q0.25, ymax=q0.75, fill = og_mce),
  #             alpha = 0.2) +
  geom_line(
    alpha = 1, size = 0.1, linewidth = 0.2) +
  geom_point_interactive(
    aes(tooltip=tooltip, data_id = year),
    data = ~filter(.x, year%in%c(2029, 2045, 2050)), size = 0.5) +
 scale_y_continuous(
    labels=scales::label_percent(1),
    oob = scales::squish)+
  scale_x_continuous(breaks = c(2030, 2040, 2050), 
                     labels = date1,
                     minor_breaks = scales::breaks_width(1)) +
  guides(color = guide_colorbar(title = "multitplicateur  \nà 1 an"),
         fill = "none",
         x = guide_axis(minor.ticks = TRUE)) +
  scale_color_distiller(palette="Greens", direction = 1)+
  facet_grid(
    cols=vars(variable), rows=vars(og_mce),
    labeller = labeller(
      variable = c(ibc = "Cumul des impulsions", og = "Ecart de production"),
      og_mce = ajustl) )+
  theme_ofce(
    legend.position.inside = c(0.95, 0.05),
    legend.direction = "horizontal",
    legend.justification = c(1,0),
    legend.key.width = unit(12, "pt"),
    legend.text = element_text(hjust=.5),
    legend.text.position = "bottom",
    axis.text.x = element_text(hjust=0, size = rel(0.9)),
    legend.title = marquee::element_marquee(vjust = .5),
    panel.spacing = unit(5, "pt") )+
  xlab(NULL)+
  ylab("Cumul des impulsions")

girafy(gminog, 1.5)

## *Backloading* versus *frontloading*

On simule ici deux variations autour du PSMT. Le *frontloading* consiste à accroître la première impulsion négative, en 2025, à 1,4 point de PIB au lieu de 0,7 et à arrêter l’ajustement une année plus tôt (en 2028 au lieu de 2029). Le *backloading* est la manœuvre inverse : une impulsion neutre en 2025 et une année supplémentaire d’ajustement à 0,7% en 2030. On applique ensuite la politique optimale comme pour le graphique précédent.

Le PSMT surajustant le *backloading*, permet de se rapprocher de l’optimum dans la plupart des cas. Le *backloading* impose cependant un effort un peu plus brutal ensuite.

In [ ]:
readRDS(pathify("/ajust/sensibilite.rds")) |> 
  filter(year>=2029, year<=2045, mult=="fixe", ib %in% c("frontload", "backload"), objective == "og") |> 
  mutate(
    ib = factor(ib, c("optimal", "psmt", "frontload", "backload")),
    quantile = str_c("q", quantile)) |>
  select(year, quantile, dstar, ecstar, mult, ib, value = ibc, objective) |> 
  pivot_wider(names_from = quantile, values_from = value) |>
  arrange(mult, objective, dstar, ib, year) |> 
  group_by(mult, objective, ecstar, dstar, ib) |> 
  mutate(
    libc1 = str_c("min: ",round(min(q0.5[year>=2029])*100, 2), "%"),
    libc2 = str_c("final: ", round(q0.5[year==max(year)]*100, 2), "%"),
    libc = ifelse(year==2030, str_c(libc1, "   \n", libc2), NA_character_)) |> 
  ungroup() |> 
  ggplot() +
  aes(y =q0.5, ymin=q0.25, ymax =q0.75, 
      x = factor(dstar, label = c("100%", "110%", "120%")), color = year, group = year) +
  geom_pointrange(
    position = position_dodge(0.5),
    alpha = 1, size = 0.1, linewidth = 0.2) +
  marquee::geom_marquee(
    aes(label=libc, y=-0.055),
    color = "black", hjust = .5,
    size.unit = "pt",
    size = small_font_size )+
  scale_color_distiller(palette="Blues", direction = -1, name = NULL)+
  scale_y_continuous(
    limits = c(-0.06, -0.02),
    breaks=scales::breaks_width(0.01),
    labels=scales::label_percent(1),
    oob = scales::squish)+
  guides(color = guide_legend(nrow = 1)) +
  theme_ofce(
    legend.position = "bottom",
    legend.direction = "horizontal",
    legend.key.width = unit(24, "pt"),
    legend.text.position = "bottom",
    legend.title = marquee::element_marquee(vjust = 0.9),
    panel.spacing = unit(12, "pt")
  )+
  facet_grid(
    rows=vars(ecstar), cols=vars(ib),
    labeller = as_labeller(
      c("0.005"="Ecart critique -0.5%", "0"="Ecart critique nul", "-0.005"="Ecart critique 0.5%",
        psmt = "PSMT", optimal = "Optimal", frontload = "Frontload", backload = "Backload")
    )) +
  xlab("Cible de dette")+
  ylab("Cumul de l'écart de production")

## Une cible alternative : stabilisation de la dette

## Annexe : équations du modèle *debtwatch*

L’équation de retour à l’équilibre de l’écart de production est écrite comme un modèle à correction d’erreur (MCE). Les impulsions jouent sur l’écart de production. Le multiplicateur instantané est défini par le paramètre $og_{dep}$ ou $og_{po}$. Le multiplicateur complet (ou de long terme) dépend des autres paramètres du MCE, $og_{lag}$ et $og_{mce}$. Les valeurs utilisées pour les simulations sont : $og_{dep}=0.8$, $og_{po}=0.5$, $og_{mce}=0.3$, $og_{lag}=0.25$.

$$
\Delta og = og_{lag} \times \Delta og_{-1} -og_{mce} \times og + og_{dep} \times ib_{dep} + og_{po} \times ib_{po} + \varepsilon
$$ {#eq-og}

Un bruit est ajouté, avec une autocorrélation. $ogn_\sigma=0.015$ et $ogn_{ar}=0.9$.

$$
\Delta \varepsilon = ogn_{ar} \times \varepsilon_{-1} + \epsilon_{og} (\sim \mathcal{N}(0,ogn_{\sigma}^2))
$$

Le taux de croissance du prix du PIB (appelé inflation, on ne distingue pas prix de consommation ou autres), suit également un MCE. Le long terme est défini par la cible d’inflation et un effet de Phillips. Les paramètres sont $inf_{lag}=0$, $inf_{mce}=0.68$, $inf_{tcho}=0.15$.

$$
\Delta\dot{p} = inf_{lag} \times d\dot p_{-1} - inf_{mce} \times (\dot p - inf_{star} - inf_{tcho} \times (u - nairu)) +\eta
$$ Comme pour l’écart de production, on ajoute une perturbation aléatoire. $inf_{\sigma}=0.01$, $infn_{ar}=0.0$. $$
\Delta \eta = infn_{ar} \times \eta_{-1} + \epsilon_{inf} (\sim \mathcal{N}(0,inf_{sigma}^2))
$$ La dynamique de la dette suit une équation habituelle. $t_{aut}$ assure l’équilibre comptable et prend en compte les recettes non fiscales.

$$
\Delta dette = -s_p + int - t_{aut} \times vpib + one_{off}
$$

Le taux de marché est défini comme la somme de l’inflation, de la croissance potentielle, d’une prime de risque $ec$ et d’un terme qui dépend de l’écart de la dette à une dette de référence $d^{**}$. Dans les simulations $r_{dette}$ est nul.

$$
r_{inst} = g_{pot} + \dot{p} + ec + r_{dette} \times (d - d^{**})
$$

La prime de risque suit un MCE, $ec_{mce}=0.5$.

$$
\Delta ec = -ec_{mce} \times (ec_{-1} - ec^*)
$$

Le taux moyen pondéré de la dette publique ou taux apparent est la moyenne entre le taux moyen pondéré antérieur et le taux courant, comme approximation de tranches de dettes de maturité égale et de différentes dates d’émission. $r_{mat}=9$.

$$
r_{app} = 1/r_{mat} \times r_{inst, -1} + (1-1/r_{mat}) \times r_{app, -1}
$$

La règle linéaire qui définit les impulsions est ($tpo_{s^*}=0.25$, $tpo_{og}=-0.25$, $tpo_{d^*}=-0.1$:

$$
ib_{lr} = tpo_{s^*} \times (spp - s^*) + tpo_{d^*} \times (d- d^*) + tpo_{og}\times og
$$

Le chômage suit un MCE dont la cible est le nairu et qui est perturbé par l’écart de production, le coefficient d’Okun à court terme est définit par $tcho_{okun}=0.5$ et unitaire à long terme (i.e. en cas de choc persistant sur l’écart de production). $tcho_{mce}=0.27$.

$$
\Delta tcho = - tcho_{okun} \times dog - tcho_{mce} \times (tcho+og-nairu)
$$

Le nairu s’ajuste progressivement à sa cible, afin de lisser les évolutions. $nairu_{mce}=0.2$

$$
\Delta nairu = - nairu_{mce} \times (nairu_{-1} - nairu^*)
$$